In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [2]:
os.makedirs('artifacts/figures', exist_ok=True)

In [3]:
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [4]:
set_seed(SEED)

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
print(f"device: {device}")

device: cpu


In [7]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [8]:
# Загрузка EMNIST (Balanced)
full_train_dataset = datasets.EMNIST(root='./data', split='balanced', train=True, download=True, transform=transform)
test_dataset = datasets.EMNIST(root='./data', split='balanced', train=False, download=True, transform=transform)

100%|██████████| 562M/562M [00:03<00:00, 162MB/s]


In [9]:
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [10]:
x_batch, y_batch = next(iter(train_loader))
print(f"Размер батча (X): {x_batch.shape}")
print(f"Размер батча (Y): {y_batch.shape}")
print(f"Диапазон значений X: [{x_batch.min():.4f}, {x_batch.max():.4f}]")
print(f"Количество классов в EMNIST balanced:{(len(full_train_dataset.classes))}")

Размер батча (X): torch.Size([128, 1, 28, 28])
Размер батча (Y): torch.Size([128])
Диапазон значений X: [-1.0000, 1.0000]
Количество классов в EMNIST balanced:47


In [11]:
class MLP(nn.Module):
    def __init__(self, use_dropout=False, use_bn=False, dropout_p=0.3):
        super().__init__()
        self.flatten = nn.Flatten()

        layers = []
        # Input: 28x28 = 784
        layers.append(nn.Linear(784, 256))
        if use_bn: layers.append(nn.BatchNorm1d(256))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(dropout_p))

        layers.append(nn.Linear(256, 128))
        if use_bn: layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if use_dropout: layers.append(nn.Dropout(dropout_p))

        # Output: 47 классов для EMNIST Balanced
        layers.append(nn.Linear(128, 47))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(self.flatten(x))

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        loss = criterion(outputs, y)

        running_loss += loss.item() * X.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

In [12]:
runs_results = []

def run_experiment(exp_id, use_dropout, use_bn, optimizer_name, lr,
                   momentum=0, weight_decay=0, max_epochs=10, patience=None):
    set_seed(SEED)
    model = MLP(use_dropout=use_dropout, use_bn=use_bn).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    best_val_acc = 0.0
    best_val_loss = float('inf')
    epochs_trained = 0
    epochs_no_improve = 0
    best_state = None

    for epoch in range(max_epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        v_loss, v_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        epochs_trained += 1

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_val_loss = v_loss
            epochs_no_improve = 0
            best_state = model.state_dict()
        else:
            epochs_no_improve += 1

        if patience and epochs_no_improve >= patience:
            print(f"[{exp_id}] Early stopping на эпохе {epoch+1}")
            break

    config_str = f"256-128/ReLU/DO:{use_dropout}/BN:{use_bn}"
    runs_results.append({
        'experiment_id': exp_id, 'dataset': 'EMNIST', 'seed': SEED,
        'model_summary': config_str, 'optimizer': optimizer_name,
        'lr': lr, 'momentum': momentum, 'weight_decay': weight_decay,
        'epochs_trained': epochs_trained, 'best_val_accuracy': round(best_val_acc, 4),
        'best_val_loss': round(best_val_loss, 4)
    })

    print(f"[{exp_id}] Завершено. Лучшая val_acc: {best_val_acc:.4f}")
    if best_state:
        model.load_state_dict(best_state)
    return model, history

Регуляризация (E1-E4)

In [13]:
print("--- Запуск Части A: Регуляризация ---")
# E1 (Base)
_, hist_e1 = run_experiment('E1', False, False, 'Adam', 1e-3, max_epochs=10)
# E2 (Dropout)
_, hist_e2 = run_experiment('E2', True, False, 'Adam', 1e-3, max_epochs=10)
# E3 (BatchNorm)
_, hist_e3 = run_experiment('E3', False, True, 'Adam', 1e-3, max_epochs=10)


best_reg_use_bn = True
best_reg_use_do = False


best_model, hist_e4 = run_experiment('E4', best_reg_use_do, best_reg_use_bn,
                                     'Adam', 1e-3, max_epochs=20, patience=3)

test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss())
print(f"\n[FINAL] E4 Test Accuracy: {test_acc:.4f}")


torch.save(best_model.state_dict(), 'artifacts/best_model.pt')
best_config = {
    'experiment_id': 'E4', 'dataset': 'EMNIST', 'seed': SEED,
    'architecture': 'Flatten -> Linear(784,256) -> BatchNorm1d -> ReLU -> Linear(256,128) -> BatchNorm1d -> ReLU -> Linear(128,47)',
    'optimizer': 'Adam', 'lr': 1e-3, 'epochs_trained': len(hist_e4['val_loss']),
    'early_stopping_patience': 3, 'test_accuracy': test_acc
}
with open('artifacts/best_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

--- Запуск Части A: Регуляризация ---
[E1] Завершено. Лучшая val_acc: 0.8359
[E2] Завершено. Лучшая val_acc: 0.8194
[E3] Завершено. Лучшая val_acc: 0.8494
[E4] Early stopping на эпохе 12
[E4] Завершено. Лучшая val_acc: 0.8494

[FINAL] E4 Test Accuracy: 0.8419


Оптимизация (O1-O3)

In [14]:
print("\n--- Запуск Части B: Оптимизация ---")
# O1 (LR слишком большой)
_, hist_o1 = run_experiment('O1', best_reg_use_do, best_reg_use_bn, 'Adam', 1e-1, max_epochs=6)
# O2 (LR слишком маленький)
_, hist_o2 = run_experiment('O2', best_reg_use_do, best_reg_use_bn, 'Adam', 1e-5, max_epochs=6)
# O3 (SGD + Momentum + Weight Decay)
_, hist_o3 = run_experiment('O3', best_reg_use_do, best_reg_use_bn, 'SGD', 1e-2, momentum=0.9, weight_decay=1e-4, max_epochs=12)


--- Запуск Части B: Оптимизация ---
[O1] Завершено. Лучшая val_acc: 0.8019
[O2] Завершено. Лучшая val_acc: 0.6393
[O3] Завершено. Лучшая val_acc: 0.8476


In [15]:
df_runs = pd.DataFrame(runs_results)
df_runs.to_csv('artifacts/runs.csv', index=False)

In [16]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_e4['train_loss'], label='Train Loss')
plt.plot(hist_e4['val_loss'], label='Val Loss')
plt.title('E4: Loss Curve')
plt.xlabel('Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist_e4['train_acc'], label='Train Acc')
plt.plot(hist_e4['val_acc'], label='Val Acc')
plt.title('E4: Accuracy Curve')
plt.xlabel('Epochs')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/figures/curves_best.png')
plt.close()

In [17]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_o1['train_loss'], label='O1 (lr=1e-1) Train Loss', color='red')
plt.plot(hist_o2['train_loss'], label='O2 (lr=1e-5) Train Loss', color='blue')
plt.title('Extremes: Train Loss')
plt.xlabel('Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist_o1['val_acc'], label='O1 (lr=1e-1) Val Acc', color='red')
plt.plot(hist_o2['val_acc'], label='O2 (lr=1e-5) Val Acc', color='blue')
plt.title('Extremes: Validation Accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.tight_layout()
plt.savefig('artifacts/figures/curves_lr_extremes.png')
plt.close()